# 🖼️ Datenaugmentierung für Transfer-Learning – die Bilder-Werkstatt (RGB · 224×224)

**Worum geht es?** Wir bauen eine kleine Gradio-App, mit der du das **Prinzip der Datenaugmentierung**
hands-on erlebst – diesmal zugeschnitten auf **echtes Transfer-Learning** mit **VGG16** bzw. **MobileNet**.
Du lädst Bilder, schiebst an Reglern, und erzeugst daraus einen größeren, augmentierten Datensatz
im **224×224-RGB-Format**, fertig abgelegt als **Bildordner** für `torchvision.datasets.ImageFolder`.

> **Was sich gegenüber der Graustufen-/CSV-Version geändert hat**
> | früher (CNN-from-scratch) | jetzt (Transfer-Learning) |
> |---|---|
> | Graustufe, 1 Kanal, kleine Quadrate | **RGB, 3 Kanäle, 224×224** (VGG16/MobileNet-Input) |
> | Ausgabe = eine **CSV** mit Pixel-Features | Ausgabe = **Bilddateien** im `ImageFolder`-Baum |
> | kein Train/Test-Split | **Quell-basierter Train/Test-Split** (leckagefrei) |
> | 7 Transformationen | **8** (zusätzlich `ColorJitter` – sinnvoll erst in Farbe) |

---

### Der rote Faden – drei Ideen, die hier zusammenkommen

**1️⃣ Ein Datenpunkt *ist* ein Bild – und bleibt ein Bild.**
Beim Transfer-Learning füttern wir das vortrainierte Netz nicht mit CSV-Zeilen, sondern mit echten
**224×224-RGB-Bildern**. Deshalb speichern wir den augmentierten Datensatz direkt als Bildordner –
genau die Struktur, die `ImageFolder` erwartet:

```
Datasets/Train/<klasse>/*.png
Datasets/Test/<klasse>/*.png
```

**2️⃣ Augmentierung = mehr Daten ohne neue Daten.**
Aus *einem* Quellbild machen wir durch **label-erhaltende** Transformationen viele Varianten.
Das Label bleibt dasselbe – ein gedrehter Apfel ist ein Apfel. So lernt das Modell **Invarianzen**
und überanpasst weniger. Beim Transfer-Learning mit kleinem Datensatz ist das oft *der* entscheidende Hebel.

**3️⃣ Augmentieren nur auf den Trainingsdaten!**
Test-/Validierungsbilder werden **nicht** augmentiert, sondern nur deterministisch auf 224×224 gebracht
(`Resize(256)` → `CenterCrop(224)`, das klassische ImageNet-Eval-Rezept). Sonst entsteht **Datenleckage**
und die gemessene Genauigkeit lügt. Genau diese Trennung baut die App von vornherein ein.


### Was die App können wird
- 📂 Quellbilder laden – **flach** (`apfel.png`, `apfel_001.png`, …) **oder** als Unterordner (`apfel/…`)
- 🔢 Klassen **label-encodieren** (Text → Integer) und in einer **JSON** sichern
- ✂️ **Train/Test-Split auf Quell-Ebene** (Anteil per Regler) – leckagefrei
- 🎚️ 8 Augmentierungen über Regler/Checkboxen steuern – mit **RGB-Live-Vorschau**
- 🔀 die **Reihenfolge** der Transformationen festlegen (zählt!)
- 💾 Datensatz als **224×224-RGB-Bilder** in `Datasets/Train|Test/<klasse>/` speichern
- 🧾 zusätzlich **`manifest.csv`** (Pfad, Klasse, Label, Split) und **`class_index.json`** schreiben
- ♻️ die JSON-Konfiguration später **wieder laden** und den Zustand herstellen
- 🚀 am Ende: fertiger **`ImageFolder`-/`DataLoader`-Codeblock** für VGG16/MobileNet


## 0 · Setup

Wir brauchen `torch`, `torchvision`, `gradio`, `pillow`, `numpy`, `pandas`, `matplotlib`.
Falls etwas fehlt, die nächste Zelle einmal ausführen (danach Kernel ggf. neu starten).

In [ ]:
# Bei Bedarf einkommentieren:
# %pip install torch torchvision gradio pillow numpy pandas matplotlib

import os, io, json, base64, re, shutil, random
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch
from torchvision import transforms

import matplotlib
matplotlib.use("Agg")          # backend ohne Fenster - wichtig fuer Gradio im Notebook
import matplotlib.pyplot as plt

import gradio as gr

ZIEL_SIZE = 224                # Kantenlaenge fuer VGG16 / MobileNet
BILD_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".webp")

print("torch      :", torch.__version__)
print("gradio     :", gr.__version__)
print("Ziel-Format:", f"{ZIEL_SIZE}x{ZIEL_SIZE} RGB")
print("Alles bereit \u2714")

## 1 · Beispielbilder anlegen (farbige Formen, nativ 224×224)

Damit das Notebook sofort läuft, **zeichnet** die nächste Zelle vier farbige Formen direkt in voller
Auflösung **224×224×3** – es wird **nichts hochskaliert**. Pro Klasse entstehen mehrere leicht verschiedene
Varianten (andere Position, Größe, Hintergrund, Linienrichtung), damit der **Train/Test-Split von Anfang an
leckagefrei** funktioniert.

Klassen & Signaturfarben: 🔴 `kreis`, 🟢 `dreieck`, 🟡 `rechteck`, 🔵 `linie`. Der **Dateiname ist die Klasse**;
die laufende Nummer (`_000`, `_001`, …) wird beim Laden automatisch abgeschnitten.

> ⚠️ **Didaktischer Hinweis:** Hier korreliert die Farbe perfekt mit der Klasse – ein Modell könnte also
> allein an der Farbe „schummeln". Für die *Augmentierungs-Mechanik* spielt das keine Rolle; für ein ehrliches
> Klassifikations-Experiment nimmst du echte Fotos (oder mischst die Farben pro Klasse zufällig). Für echtes
> Transfer-Learning ohnehin: eigene Fotos, möglichst viele pro Klasse.

In [ ]:
# Farbige Beispielformen DIREKT in 224x224x3 zeichnen (kein Hochskalieren!)
FARBEN = {                       # Signaturfarbe je Klasse (R, G, B)
    "kreis":    (220,  60,  60), # rot
    "dreieck":  ( 60, 180,  90), # gruen
    "rechteck": (235, 200,  60), # gelb
    "linie":    ( 70, 140, 230), # blau
}
BEISPIEL_ORDNER     = "Datasets/Beispiel_Formen"
BEISPIEL_PRO_KLASSE = 8           # mehrere Varianten je Klasse -> leckagefreier Split moeglich


def _zeichne_form(klasse, groesse=ZIEL_SIZE, seed=0):
    """Zeichnet EINE farbige Form nativ in (groesse x groesse) RGB, mit leichter Variation."""
    rng = random.Random(seed)
    bg = tuple(rng.randint(15, 60) for _ in range(3))          # dunkler, leicht variierender Grund
    img = Image.new("RGB", (groesse, groesse), bg)
    d = ImageDraw.Draw(img)
    farbe = FARBEN[klasse]
    pad = rng.randint(20, 55)
    box = [pad, pad, groesse - pad, groesse - pad]
    if klasse == "kreis":
        d.ellipse(box, fill=farbe)
    elif klasse == "rechteck":
        d.rectangle(box, fill=farbe)
    elif klasse == "dreieck":
        d.polygon([(groesse // 2, pad), (pad, groesse - pad), (groesse - pad, groesse - pad)], fill=farbe)
    elif klasse == "linie":
        breite = rng.randint(10, 24)
        if rng.random() < 0.5:
            d.line([(pad, pad), (groesse - pad, groesse - pad)], fill=farbe, width=breite)
        else:
            d.line([(pad, groesse - pad), (groesse - pad, pad)], fill=farbe, width=breite)
    return img


def schreibe_beispielbilder(ordner=BEISPIEL_ORDNER, n_pro_klasse=BEISPIEL_PRO_KLASSE):
    """Erzeugt n_pro_klasse farbige 224x224-Varianten je Klasse; Dateiname = Klasse + lauf. Nummer."""
    if os.path.isdir(ordner):
        shutil.rmtree(ordner)
    os.makedirs(ordner, exist_ok=True)
    for ki, klasse in enumerate(FARBEN):
        for i in range(n_pro_klasse):
            img = _zeichne_form(klasse, ZIEL_SIZE, seed=1000 * ki + i)
            img.save(os.path.join(ordner, f"{klasse}_{i:03d}.png"))
    return ordner


schreibe_beispielbilder()
_dateien = sorted(os.listdir(BEISPIEL_ORDNER))
print(f"Geschrieben: {len(_dateien)} Bilder ({BEISPIEL_PRO_KLASSE} je Klasse) in '{BEISPIEL_ORDNER}/'")
_probe = Image.open(os.path.join(BEISPIEL_ORDNER, _dateien[0]))
print("Beispiel:", _dateien[0], "->", _probe.size, _probe.mode, "(nativ gezeichnet, nicht skaliert)")

## 2 · Laden & Label-Encoding

**Was passiert hier?**
- Alle Bilder werden als **RGB** (`mode="RGB"`, 3 Kanäle, Werte 0–255) geladen.
- Zwei Quell-Layouts werden automatisch erkannt:
  - **Unterordner** vorhanden → ImageFolder-Stil: *Ordnername = Klasse* (mehrere Bilder pro Klasse).
  - sonst **flach** → *Dateiname = Klasse*; eine angehängte laufende Nummer (`apfel_001`, `apfel-2`)
    wird abgeschnitten, damit du **mehrere Bilder pro Klasse** ablegen kannst.
- Die Klassen werden **sortiert** und auf Integer abgebildet (`label_encoding`), z. B.
  `{"dreieck": 0, "kreis": 1, "linie": 2, "rechteck": 3}`.
- **Anders als bei der CSV-Version** müssen die Bilder *nicht* quadratisch oder gleich groß sein –
  jede beliebige Auflösung/Seitenverhältnis ist ok, weil später ohnehin auf 224×224 standardisiert wird.

In [ ]:
def _klasse_aus_dateiname(stem):
    """Schneidet eine angehaengte laufende Nummer ab: 'apfel_001' -> 'apfel', 'birne-2' -> 'birne'."""
    return re.sub(r"[_-]\d+$", "", stem)


def lade_quelle(ordner):
    """Liest Quellbilder als RGB. Zwei Modi (automatisch erkannt):
       (a) Unterordner vorhanden -> Unterordnername = Klasse
       (b) flach                 -> Dateiname (ohne lauf. Nummer) = Klasse
    Rueckgabe: (items, encoding) mit items = Liste von (klasse, PIL.Image 'RGB')."""
    if not os.path.isdir(ordner):
        raise FileNotFoundError(f"Ordner nicht gefunden: {ordner}")
    subdirs = [d for d in sorted(os.listdir(ordner))
               if os.path.isdir(os.path.join(ordner, d))]
    items = []
    if subdirs:                                  # ImageFolder-Stil
        for d in subdirs:
            for fn in sorted(os.listdir(os.path.join(ordner, d))):
                if fn.lower().endswith(BILD_EXTS):
                    bild = Image.open(os.path.join(ordner, d, fn)).convert("RGB")
                    items.append((d, bild))
    else:                                        # flach: Dateiname = Klasse
        for fn in sorted(os.listdir(ordner)):
            if fn.lower().endswith(BILD_EXTS):
                klasse = _klasse_aus_dateiname(os.path.splitext(fn)[0])
                bild = Image.open(os.path.join(ordner, fn)).convert("RGB")
                items.append((klasse, bild))
    if not items:
        raise ValueError("Keine Bilddateien gefunden (png/jpg/jpeg/bmp/webp).")

    klassen = sorted({k for k, _ in items})
    encoding = {k: i for i, k in enumerate(klassen)}
    return items, encoding


# kurzer Test
_items, _enc = lade_quelle(BEISPIEL_ORDNER)
from collections import Counter
_zaehl = Counter(k for k, _ in _items)
print("Klassen + Encoding:", _enc)
print("Bilder pro Klasse :", dict(_zaehl))
print("Anzahl Quellbilder:", len(_items))

## 2b · Train/Test-Split – *vor* dem Augmentieren ✂️

Das ist der wichtigste konzeptionelle Unterschied zur alten Version. Wir teilen die **Quellbilder**
pro Klasse in Train und Test auf – und augmentieren **nur** die Trainingsquellen. Die Testbilder
durchlaufen später ausschließlich das deterministische Eval-Rezept (`Resize(256)`→`CenterCrop(224)`).

> 🔒 **Warum auf Quell-Ebene splitten?** Würde man erst augmentieren und *danach* zufällig splitten,
> landeten Varianten **desselben** Originalbilds in Train *und* Test → **Datenleckage**. Das Modell
> „kennt" die Testbilder dann quasi schon, und die Genauigkeit ist geschönt.

> 🧪 **Einzelbild-Fall:** Hat eine Klasse nur **ein** Quellbild (wie im eingebetteten Demo), ist ein
> leckagefreier Split unmöglich. Standard: das Bild geht nach **Train**, Test bleibt für die Klasse leer
> (mit Hinweis). Für eine reine Vorführung des Ordnerbaums gibt es einen ausdrücklich beschrifteten
> **Demo-Modus**, der dieselbe Quelle in beide Splits legt – *bewusst mit Leckage, nur zum Zeigen*.

In [ ]:
def split_quelle(items, test_anteil, seed, demo_leckage=False):
    """Teilt QUELLBILDER pro Klasse in Train/Test. Augmentiert wird spaeter nur Train.
    Rueckgabe: (train_items, test_items, warnungen)."""
    rng = random.Random(seed)
    nach_klasse = {}
    for k, im in items:
        nach_klasse.setdefault(k, []).append(im)

    train_items, test_items, warnungen = [], [], []
    for k in sorted(nach_klasse):
        ims = nach_klasse[k][:]
        rng.shuffle(ims)
        n = len(ims)
        if n == 1:
            train_items.append((k, ims[0]))
            if demo_leckage:
                test_items.append((k, ims[0]))
                warnungen.append(f"'{k}': nur 1 Quellbild \u2192 DEMO-LECKAGE (Quelle in Train UND Test)")
            else:
                warnungen.append(f"'{k}': nur 1 Quellbild \u2192 Test bleibt leer (lege \u22652 Bilder an)")
            continue
        n_test = min(max(1, round(test_anteil * n)), n - 1)   # mind. 1 Test, mind. 1 Train
        for im in ims[:n_test]:
            test_items.append((k, im))
        for im in ims[n_test:]:
            train_items.append((k, im))
    return train_items, test_items, warnungen


# kurzer Test (eingebettetes Demo: 1 Bild/Klasse)
_tr, _te, _w = split_quelle(_items, test_anteil=0.2, seed=42, demo_leckage=False)
print(f"Train-Quellen: {len(_tr)} | Test-Quellen: {len(_te)}")
for _z in _w:
    print("  \u26a0\ufe0f", _z)

## 3 · Die acht Augmentierungen

Wir nutzen **`torchvision.transforms`** als Engine – jede Transformation ist eine kleine, benennbare
Operation. Manche Regler steuern eine **Stärke**, andere eine **Wahrscheinlichkeit** – steht jeweils dabei.

| Transformation | Regler steuert | Wirkung |
|---|---|---|
| `RandomResizedCrop` | **Stärke** (kleinste Crop-Skala) | schneidet zufälligen Bereich aus, skaliert **auf 224×224** → Zoom/Ausschnitt |
| `RandomHorizontalFlip` | **Checkbox** (p = 0,5) | spiegelt links↔rechts |
| `RandomVerticalFlip` | **Checkbox** (p = 0,5) | spiegelt oben↔unten |
| `RandomRotation` | **Stärke** (max. Grad) | dreht um zufälligen Winkel ±Grad |
| `RandomPerspective` | **Stärke** (Verzerrung) | „kippt" das Bild perspektivisch (p = 0,5) |
| `ColorJitter` | **Stärke** (Jitter) | variiert Helligkeit/Kontrast/Sättigung – **nur in RGB sinnvoll** |
| `RandomAdjustSharpness` | **Stärke** (Faktor; 1 = neutral) | <1 weichzeichnen, >1 schärfen |
| `RandomAutocontrast` | **Wahrscheinlichkeit** | streckt den Kontrast auf vollen Bereich |

> ✅ **Ausgabe ist garantiert 224×224×3.** Egal welche Regler aktiv sind, am Ende der Pipeline steht
> immer ein `Resize((224,224))` als Sicherheitsnetz. `RandomResizedCrop` liefert ohnehin schon 224×224;
> ist es ausgeschaltet, sorgt das finale `Resize` für die korrekte Modell-Eingabegröße. Die Feature-Geometrie
> bleibt also stabil – nötig, weil VGG16/MobileNet eine feste Eingabegröße erwarten.

> 🔄 **Reihenfolge zählt!** Jede Transformation hat eine **Reihenfolge-Nummer**. Das `Compose`-Objekt
> wird genau in dieser Reihenfolge gebaut: *erst drehen, dann kippen* ≠ *erst kippen, dann drehen*
> (eigenes Experiment am Ende).

In [ ]:
# --- Metadaten aller Transformationen (steuert spaeter automatisch die UI) -----------
# typ: "scale" -> Slider fuer Staerke ;  "flip" -> Checkbox (an => p=0.5) ;  "prob" -> Slider fuer p
TRANSFORM_DEFS = [
    dict(name="RandomResizedCrop",    typ="scale", label="ResizedCrop \u00b7 Skala (1 = aus)",
         lo=0.2, hi=1.0, step=0.05, default=1.0, neutral=1.0,
         hilfe="1.0 = kein Crop, kleiner = staerkerer Zoom-Ausschnitt"),
    dict(name="RandomHorizontalFlip", typ="flip",  label="\u2194 H-Flip (p = 0,5)",
         default=False, hilfe="an => 50% der Bilder horizontal gespiegelt"),
    dict(name="RandomVerticalFlip",   typ="flip",  label="\u2195 V-Flip (p = 0,5)",
         default=False, hilfe="an => 50% der Bilder vertikal gespiegelt"),
    dict(name="RandomRotation",       typ="scale", label="Rotation \u00b7 max \u00b0 (0 = aus)",
         lo=0.0, hi=180.0, step=5.0, default=0.0, neutral=0.0,
         hilfe="dreht zufaellig um \u00b1Winkel; 0 = aus"),
    dict(name="RandomPerspective",    typ="scale", label="Perspektive \u00b7 Verzerrung (0 = aus)",
         lo=0.0, hi=1.0, step=0.05, default=0.0, neutral=0.0,
         hilfe="0 = aus; groesser = staerkere Verkippung (p = 0,5)"),
    dict(name="ColorJitter",          typ="scale", label="ColorJitter \u00b7 Staerke (0 = aus)",
         lo=0.0, hi=0.6, step=0.05, default=0.0, neutral=0.0,
         hilfe="variiert Helligkeit/Kontrast/Saettigung; 0 = aus (nur in RGB sinnvoll)"),
    dict(name="RandomAdjustSharpness",typ="scale", label="Sharpness \u00b7 Faktor (1 = neutral)",
         lo=0.0, hi=4.0, step=0.25, default=1.0, neutral=1.0,
         hilfe="<1 weichzeichnen, 1 unveraendert, >1 schaerfen"),
    dict(name="RandomAutocontrast",   typ="prob",  label="Autocontrast \u00b7 p (0 = aus)",
         lo=0.0, hi=1.0, step=0.1, default=0.0, neutral=0.0,
         hilfe="Wahrscheinlichkeit, den Kontrast voll auszureizen"),
]
NAMES = [d["name"] for d in TRANSFORM_DEFS]


def baue_compose(spec):
    """spec = Liste von dicts {name, enabled, order, value}.
    Baut ein torchvision-Compose in der per 'order' gewuenschten Reihenfolge auf RGB-PIL.
    Am Ende steht IMMER ein Resize auf 224x224 -> Ausgabe garantiert modellkonform."""
    aktive = sorted([t for t in spec if t["enabled"]], key=lambda t: t["order"])
    tfs = []
    for t in aktive:
        n, v = t["name"], t["value"]
        if n == "RandomResizedCrop":
            tfs.append(transforms.RandomResizedCrop(
                size=(ZIEL_SIZE, ZIEL_SIZE), scale=(float(v), 1.0), antialias=True))
        elif n == "RandomHorizontalFlip":
            tfs.append(transforms.RandomHorizontalFlip(p=0.5 if v else 0.0))
        elif n == "RandomVerticalFlip":
            tfs.append(transforms.RandomVerticalFlip(p=0.5 if v else 0.0))
        elif n == "RandomRotation":
            tfs.append(transforms.RandomRotation(degrees=float(v), fill=0))
        elif n == "RandomPerspective":
            tfs.append(transforms.RandomPerspective(distortion_scale=float(v), p=0.5, fill=0))
        elif n == "ColorJitter":
            tfs.append(transforms.ColorJitter(brightness=float(v), contrast=float(v),
                                              saturation=float(v)))
        elif n == "RandomAdjustSharpness":
            tfs.append(transforms.RandomAdjustSharpness(sharpness_factor=float(v), p=1.0))
        elif n == "RandomAutocontrast":
            tfs.append(transforms.RandomAutocontrast(p=float(v)))
    # Sicherheitsnetz: immer auf Modell-Eingabegroesse bringen
    tfs.append(transforms.Resize((ZIEL_SIZE, ZIEL_SIZE), antialias=True))
    return transforms.Compose(tfs)


def eval_transform():
    """Deterministisches Test-/Eval-Rezept (KEINE Augmentierung): klassisches ImageNet-Resize+CenterCrop."""
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(ZIEL_SIZE),
    ])


print("Definierte Transformationen:", NAMES)

## 4 · Augmentieren, Bildordner, Manifest & JSON

Hier sind die „Arbeitspferde":
- **`augmentiere`** – wendet das Compose auf ein RGB-Bild an, liefert ein `uint8`-Array `(224, 224, 3)`.
- **`erzeuge_datensatz`** – splittet Quellen, augmentiert **nur Train** (`n_pro_quelle` Varianten je Quelle),
  bringt **Test** deterministisch auf 224×224, und **schreibt alles als Bilddateien** in den `ImageFolder`-Baum.
- **`manifest.csv` + `class_index.json`** – schlanke Metadaten für die nachgelagerte Trainings-App.
- **`zustand_dict` / `lade_zustand`** – die **JSON** mit Encoding, Größe, Split-Anteil, allen Parametern
  und der Reihenfolge. Transformationen werden als **Daten** gespeichert (kein pickle): lesbar, sicher,
  exakt reproduzierbar.

> 🧾 Wir speichern **keine** Pixel-CSV mehr. 224×224×3 = 150 528 Werte pro Bild – das gehört in
> Bilddateien, nicht in Tabellenspalten. Die nachgelagerte Trainings-App liest den Ordner direkt mit
> `ImageFolder` (siehe letzte Zelle).

In [ ]:
def augmentiere(bild_pil, compose):
    """PIL 'RGB' -> augmentiertes uint8-Array (224, 224, 3), Werte 0..255."""
    aug = compose(bild_pil)
    return np.array(aug.convert("RGB"), dtype=np.uint8)


def _leere_splits(basis):
    """Loescht NUR die Train/Test-Unterordner der Zielbasis (verhindert Mischen alter Laeufe)."""
    for sp in ("Train", "Test"):
        p = os.path.join(basis, sp)
        if os.path.isdir(p):
            shutil.rmtree(p)


def erzeuge_datensatz(items, encoding, spec, n_pro_quelle, seed,
                      test_anteil, basis="Datasets", fmt="png",
                      demo_leckage=False, leeren=True):
    """Vollstaendiger Lauf: Split -> Train augmentieren -> Test eval-only -> als Bilder speichern.
    Rueckgabe: (info_str, vorschau_galerie, metadatendateien, warnungen)."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    train_items, test_items, warnungen = split_quelle(items, test_anteil, seed, demo_leckage)

    compose = baue_compose(spec)
    ev = eval_transform()
    if leeren:
        _leere_splits(basis)

    manifest, vorschau = [], []
    zaehler = {}

    def _speichern(split, klasse, arr):
        key = (split, klasse)
        zaehler[key] = zaehler.get(key, 0) + 1
        ordner = os.path.join(basis, split, klasse)
        os.makedirs(ordner, exist_ok=True)
        dn = f"{klasse}_{zaehler[key]:04d}.{fmt}"
        pfad = os.path.join(ordner, dn)
        im = Image.fromarray(arr, "RGB")
        if fmt in ("jpg", "jpeg"):
            im.save(pfad, quality=95)
        else:
            im.save(pfad)
        manifest.append(dict(path=pfad.replace("\\", "/"), klasse=klasse,
                             label=int(encoding[klasse]), split=split))
        return arr

    # --- TRAIN: augmentieren (n_pro_quelle Varianten je Quellbild) ---
    for klasse, bild in train_items:
        for _ in range(int(n_pro_quelle)):
            arr = augmentiere(bild, compose)
            _speichern("Train", klasse, arr)
            if len([v for v in vorschau if v[0] == "Train"]) < 6:
                vorschau.append(("Train", klasse, arr))

    # --- TEST: deterministisch, 1x je Quellbild (keine Augmentierung!) ---
    for klasse, bild in test_items:
        arr = np.array(ev(bild).convert("RGB"), dtype=np.uint8)
        _speichern("Test", klasse, arr)
        if len([v for v in vorschau if v[0] == "Test"]) < 2:
            vorschau.append(("Test", klasse, arr))

    # --- Metadaten schreiben ---
    os.makedirs(basis, exist_ok=True)
    man_df = pd.DataFrame(manifest, columns=["path", "klasse", "label", "split"])
    man_pfad = os.path.join(basis, "manifest.csv")
    man_df.to_csv(man_pfad, index=False)

    cls_pfad = os.path.join(basis, "class_index.json")
    with open(cls_pfad, "w", encoding="utf-8") as f:
        json.dump(encoding, f, indent=2, ensure_ascii=False)

    n_train = int((man_df["split"] == "Train").sum())
    n_test  = int((man_df["split"] == "Test").sum())
    info = (f"\u2705 {n_train} Train- + {n_test} Test-Bilder \u00b7 {len(encoding)} Klassen "
            f"\u00b7 {ZIEL_SIZE}\u00d7{ZIEL_SIZE} RGB \u00b7 abgelegt unter '{basis}/Train|Test/<klasse>/' "
            f"\u00b7 Manifest: {man_pfad}")
    galerie = [(arr, f"{sp}: {kl}") for sp, kl, arr in vorschau]
    return info, galerie, [man_pfad, cls_pfad], warnungen


def zustand_dict(encoding, spec, n_pro_quelle, seed, test_anteil, fmt, demo_leckage):
    """Vollstaendiger, serialisierbarer Zustand -> JSON."""
    return {
        "version": 2,
        "task": "transfer_learning_imagefolder",
        "label_encoding": encoding,
        "image_size": ZIEL_SIZE, "channels": 3, "color_mode": "RGB",
        "test_anteil": float(test_anteil),
        "n_pro_quelle": int(n_pro_quelle), "seed": int(seed),
        "format": fmt, "demo_leckage": bool(demo_leckage),
        "imagenet_mean": [0.485, 0.456, 0.406], "imagenet_std": [0.229, 0.224, 0.225],
        "transforms": spec,        # Liste von {name, enabled, order, value}
    }


def speichere_zustand(state, pfad):
    with open(pfad, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2, ensure_ascii=False)
    return pfad


def lade_zustand(pfad):
    with open(pfad, "r", encoding="utf-8") as f:
        return json.load(f)


print("Engine-Funktionen bereit \u2714")

## 5 · Die Gradio-App 🎛️

Das Interface ist auf **ein Browserfenster** ausgelegt:

1. **Quelle:** Ordner laden *oder* das eingebettete Beispiel nutzen. Beim Start wird das Beispiel
   automatisch geladen, du siehst sofort eine RGB-Vorschau.
2. **Links die Regler, rechts die Live-Vorschau** (Original + 3 zufällige Augmentierungen, 2×2, in Farbe).
   *Regler auf Neutralwert = Transformation aus.* Über das kleine **Reihenf.**-Feld stellst du die
   Reihenfolge ein (kleiner = früher).
3. **Datensatz erzeugen:** Bilder pro Quelle, Test-Anteil, Seed, Bildformat, Zielordner und der Button.

Galerie, Metadaten-Downloads und das Speichern/Laden der **JSON**-Konfiguration liegen darunter im
ausklappbaren Bereich.

In [ ]:
# ---------------------------------------------------------------- App-Status (global)
STATE = {"items": None, "encoding": None,
         "orders": list(range(1, len(TRANSFORM_DEFS) + 1))}

VORSCHAU_N = 3   # Original + 3 Augmentierungen -> 2x2-Raster


def _aktiv(d, v):
    if d["typ"] == "flip":
        return bool(v)
    return float(v) != float(d["neutral"])


def _spec_aus_ui(order_list, value_list):
    spec = []
    for d, od, va in zip(TRANSFORM_DEFS, order_list, value_list):
        spec.append(dict(name=d["name"], enabled=_aktiv(d, va), order=int(od), value=va))
    return spec


def aktion_reihenfolge(*orders):
    """Haelt die Reihenfolge-Felder als saubere Permutation 1..n (Tausch bei Konflikt)."""
    orders = [int(o) for o in orders]
    n = len(orders)
    prev = STATE.get("orders", list(range(1, n + 1)))
    geaendert = [i for i in range(n) if orders[i] != prev[i]]
    if len(geaendert) == 1:
        i = geaendert[0]
        ziel, alt = orders[i], prev[i]
        for j in range(n):
            if j != i and orders[j] == ziel:
                orders[j] = alt
    if sorted(orders) != list(range(1, n + 1)):
        rang = sorted(range(n), key=lambda k: (orders[k], k))
        fixed = [0] * n
        for r, k in enumerate(rang, start=1):
            fixed[k] = r
        orders = fixed
    STATE["orders"] = orders
    return [gr.update(value=o) for o in orders]


def aktion_laden(ordner):
    try:
        items, enc = lade_quelle(ordner)
        STATE.update(items=items, encoding=enc)
        from collections import Counter
        zaehl = Counter(k for k, _ in items)
        klassen = sorted(enc)
        einzel = [k for k, c in zaehl.items() if c == 1]
        info = (f"\u2705 {len(items)} Bilder \u00b7 {len(enc)} Klassen \u00b7 "
                f"Encoding: {enc} \u00b7 Bilder/Klasse: {dict(zaehl)}")
        if einzel:
            info += (f"  \u26a0\ufe0f nur 1 Bild bei: {einzel} \u2192 fuer echten Test "
                     f"\u22652 Bilder/Klasse anlegen (oder Demo-Modus aktivieren)")
        return info, gr.update(choices=klassen, value=klassen[0])
    except Exception as e:
        return f"\u274c Fehler: {e}", gr.update(choices=[], value=None)


def _fig_vorschau(klasse, spec, seed):
    items = STATE["items"]
    bild = dict(items)[klasse]
    torch.manual_seed(seed); np.random.seed(seed)
    compose = baue_compose(spec)
    fig, axes = plt.subplots(2, 2, figsize=(4.8, 5.1))
    axes = axes.ravel()
    orig_224 = np.array(eval_transform()(bild).convert("RGB"), dtype=np.uint8)
    axes[0].imshow(orig_224); axes[0].set_title("Original (224, eval)", fontsize=9); axes[0].axis("off")
    for j in range(VORSCHAU_N):
        arr = augmentiere(bild, compose)
        axes[j + 1].imshow(arr); axes[j + 1].set_title(f"Aug {j+1}", fontsize=9); axes[j + 1].axis("off")
    reihenfolge = " \u2192 ".join(t["name"].replace("Random", "")
                  for t in sorted([s for s in spec if s["enabled"]], key=lambda x: x["order"])) or "\u2014"
    fig.suptitle(f"{klasse}   \u00b7   {reihenfolge}", fontsize=9)
    fig.tight_layout()
    return fig


def aktion_vorschau(klasse, seed, *ui):
    if STATE["items"] is None:
        fig, ax = plt.subplots(figsize=(4.8, 2)); ax.axis("off")
        ax.text(0.5, 0.5, "Bitte Beispiel/Ordner laden.", ha="center", va="center")
        return fig
    if klasse is None:
        klasse = STATE["items"][0][0]
    n = len(TRANSFORM_DEFS)
    spec = _spec_aus_ui(ui[0:n], ui[n:2 * n])
    return _fig_vorschau(klasse, spec, int(seed))


def aktion_erzeugen(n_pro_quelle, test_anteil, seed, fmt, basis, demo_leckage, leeren, *ui):
    if STATE["items"] is None:
        return "\u274c Bitte zuerst Beispiel/Ordner laden.", None, None
    n = len(TRANSFORM_DEFS)
    spec = _spec_aus_ui(ui[0:n], ui[n:2 * n])
    info, galerie, meta, warn = erzeuge_datensatz(
        STATE["items"], STATE["encoding"], spec, int(n_pro_quelle), int(seed),
        float(test_anteil), basis=basis, fmt=fmt, demo_leckage=bool(demo_leckage), leeren=bool(leeren))

    state = zustand_dict(STATE["encoding"], spec, n_pro_quelle, seed, test_anteil, fmt, demo_leckage)
    cfg_pfad = os.path.join(basis, "augment_config.json")
    speichere_zustand(state, cfg_pfad)
    meta = meta + [cfg_pfad]

    if warn:
        info += "  \u26a0\ufe0f " + " | ".join(warn)
    return info, galerie, meta


def aktion_config_speichern(n_pro_quelle, test_anteil, seed, fmt, demo_leckage, *ui):
    if STATE["encoding"] is None:
        return "\u274c Bitte zuerst Beispiel/Ordner laden.", None
    n = len(TRANSFORM_DEFS)
    spec = _spec_aus_ui(ui[0:n], ui[n:2 * n])
    state = zustand_dict(STATE["encoding"], spec, n_pro_quelle, seed, test_anteil, fmt, demo_leckage)
    pfad = "augment_config.json"
    speichere_zustand(state, pfad)
    return f"\u2705 gespeichert: {pfad}", pfad


def aktion_config_laden(datei):
    """Liest JSON und stellt ALLE Bedienelemente wieder her."""
    n = len(TRANSFORM_DEFS)
    if datei is None:
        return ["\u26a0\ufe0f Keine Datei."] + [gr.update()] * (2 * n + 4)
    state = lade_zustand(datei if isinstance(datei, str) else datei.name)
    spec_map = {t["name"]: t for t in state.get("transforms", [])}
    order_upd, value_upd, orders_plain = [], [], []
    for d in TRANSFORM_DEFS:
        t = spec_map.get(d["name"], dict(order=1, value=d["default"]))
        order_upd.append(gr.update(value=int(t["order"])))
        value_upd.append(gr.update(value=t["value"]))
        orders_plain.append(int(t["order"]))
    STATE["orders"] = orders_plain
    info = (f"\u2705 geladen \u00b7 Encoding: {state.get('label_encoding')} \u00b7 "
            f"Groesse: {state.get('image_size')} \u00b7 Test-Anteil: {state.get('test_anteil')}")
    extra = [gr.update(value=state.get("n_pro_quelle", 20)),
             gr.update(value=state.get("test_anteil", 0.2)),
             gr.update(value=state.get("seed", 42)),
             gr.update(value=state.get("demo_leckage", False))]
    return [info] + order_upd + value_upd + extra

In [ ]:
CSS = """
.gradio-container {max-width: 1200px !important; margin: auto;}
#tf-row {gap: 4px !important; margin: 0 !important;}
.compact-md p {margin: 2px 0 !important;}
"""
THEME = gr.themes.Soft(spacing_size="sm", text_size="sm", radius_size="sm")

_GR_MAJOR = int(gr.__version__.split(".")[0])
_blocks_kw = {} if _GR_MAJOR >= 6 else dict(theme=THEME, css=CSS)
_launch_kw = dict(theme=THEME, css=CSS) if _GR_MAJOR >= 6 else {}

with gr.Blocks(title="Datenaugmentierung - Transfer-Learning", **_blocks_kw) as demo:

    gr.Markdown("### 🖼️ Datenaugmentierung – Bilder-Werkstatt (RGB · 224×224 · VGG16/MobileNet)",
                elem_classes="compact-md")

    # ===== Zeile 1: Quelle laden =====================================================
    with gr.Row():
        ordner_tb    = gr.Textbox(value="Datasets/Beispiel_Formen",
                                  label="📂 Quelle (Unterordner = Klasse, sonst Dateiname = Klasse)", scale=4)
        laden_btn    = gr.Button("Ordner laden", scale=1)
        beispiel_btn = gr.Button("⭐ Eingebettetes Beispiel", variant="primary", scale=1)
    lade_info = gr.Markdown(elem_classes="compact-md")

    # ===== Zeile 2: links Regler, rechts Live-Vorschau ===============================
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            gr.Markdown("**🎚️ Augmentierungen** — Regler auf Neutral = aus · *Reihenf.* kleiner = früher",
                        elem_classes="compact-md")
            order_comps, value_comps = [], []
            for d in TRANSFORM_DEFS:
                with gr.Row(elem_id="tf-row", equal_height=True):
                    od = gr.Number(value=NAMES.index(d["name"]) + 1, precision=0, label="Reihenf.",
                                   minimum=1, maximum=len(TRANSFORM_DEFS), scale=0, min_width=78)
                    if d["typ"] == "flip":
                        va = gr.Checkbox(value=d["default"], label=d["label"], scale=3)
                    else:
                        va = gr.Slider(minimum=d["lo"], maximum=d["hi"], step=d["step"],
                                       value=d["default"], label=d["label"], scale=3)
                order_comps.append(od); value_comps.append(va)
        with gr.Column(scale=1):
            vorschau_plot = gr.Plot(label="👁️ Live-Vorschau (RGB)")
            with gr.Row():
                klasse_dd     = gr.Dropdown(choices=[], label="Klasse", scale=2)
                vorschau_seed = gr.Number(value=0, precision=0, label="Seed", scale=1, min_width=80)
                wuerfel_btn   = gr.Button("🎲 würfeln", scale=0, min_width=90)

    # ===== Zeile 3: erzeugen & speichern =============================================
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            n_slider   = gr.Slider(1, 200, value=20, step=1, label="Augmentierte Bilder pro Train-Quelle")
            split_sl   = gr.Slider(0.0, 0.5, value=0.2, step=0.05, label="Test-Anteil (Quell-Split)")
            seed_num   = gr.Number(value=42, precision=0, label="Random-Seed", min_width=120)
        with gr.Column(scale=1):
            with gr.Row():
                fmt_dd     = gr.Dropdown(choices=["png", "jpg"], value="png", label="Bildformat", scale=1)
                basis_tb   = gr.Textbox(value="Datasets", label="Zielbasis (→ /Train, /Test)", scale=2)
            with gr.Row():
                demo_check = gr.Checkbox(value=False,
                             label="⚠️ Demo-Modus: Einzelbild in Train+Test (LECKAGE!)", scale=2)
                clean_check= gr.Checkbox(value=True, label="Zielordner vorher leeren", scale=1)
            erzeugen_btn = gr.Button("📦 Datensatz erzeugen", variant="primary")
    erzeugen_info = gr.Markdown(elem_classes="compact-md")

    # ===== Ausklappbar: Galerie, Downloads, JSON-Konfiguration =======================
    with gr.Accordion("🖼️ Galerie · Metadaten-Downloads · Konfiguration (JSON) speichern/laden", open=False):
        galerie_out    = gr.Gallery(label="Vorschau erzeugter Bilder", columns=4, height=240)
        download_files = gr.File(label="Metadaten (manifest.csv · class_index.json · config)",
                                 file_count="multiple")
        with gr.Row():
            cfg_save_btn = gr.Button("💾 Konfiguration speichern", scale=1)
            cfg_file_out = gr.File(label="Konfig-Download", scale=1)
            cfg_upload   = gr.File(label="Konfiguration laden (.json)", file_types=[".json"], scale=1)
        cfg_info = gr.Markdown(elem_classes="compact-md")

    # ===== Verkabelung ===============================================================
    UI_INPUTS = order_comps + value_comps                 # erst order, dann value
    vorschau_inputs = [klasse_dd, vorschau_seed] + UI_INPUTS

    laden_btn.click(aktion_laden, [ordner_tb], [lade_info, klasse_dd]).then(
        aktion_vorschau, vorschau_inputs, [vorschau_plot])
    beispiel_btn.click(lambda: "Datasets/Beispiel_Formen", None, [ordner_tb]).then(
        aktion_laden, [ordner_tb], [lade_info, klasse_dd]).then(
        aktion_vorschau, vorschau_inputs, [vorschau_plot])

    for comp in value_comps + [klasse_dd, vorschau_seed]:
        comp.change(aktion_vorschau, vorschau_inputs, [vorschau_plot])
    for comp in order_comps:
        comp.change(aktion_reihenfolge, order_comps, order_comps).then(
            aktion_vorschau, vorschau_inputs, [vorschau_plot])
    wuerfel_btn.click(lambda s: int(s) + 1, [vorschau_seed], [vorschau_seed])

    erzeugen_btn.click(
        aktion_erzeugen,
        [n_slider, split_sl, seed_num, fmt_dd, basis_tb, demo_check, clean_check] + UI_INPUTS,
        [erzeugen_info, galerie_out, download_files])
    cfg_save_btn.click(
        aktion_config_speichern,
        [n_slider, split_sl, seed_num, fmt_dd, demo_check] + UI_INPUTS,
        [cfg_info, cfg_file_out])
    cfg_upload.change(
        aktion_config_laden, [cfg_upload],
        [cfg_info] + order_comps + value_comps + [n_slider, split_sl, seed_num, demo_check])

    demo.load(aktion_laden, [ordner_tb], [lade_info, klasse_dd]).then(
        aktion_vorschau, vorschau_inputs, [vorschau_plot])

# Starten (im Notebook inline)
demo.launch(**_launch_kw, inbrowser=True)

## 6 · 🔬 Mini-Experiment: Reihenfolge zählt!

Die schönste Einsicht zum Schluss – **ganz ohne Gradio**. Wir nehmen den **Kreis** und vergleichen zwei
Pipelines mit *denselben* Transformationen, nur in **unterschiedlicher Reihenfolge**:

- **A:** erst `RandomRotation`, dann `RandomPerspective`
- **B:** erst `RandomPerspective`, dann `RandomRotation`

Bei gleichem Seed sind die Einzeloperationen identisch – aber das **Ergebnis** unterscheidet sich,
weil die zweite Operation auf einem anderen Zwischenbild arbeitet. **Transformationen kommutieren nicht.**

In [ ]:
import matplotlib.pyplot as plt
items, enc = lade_quelle(BEISPIEL_ORDNER)
kreis = dict(items)["kreis"]

spec_A = [
    dict(name="RandomRotation",    enabled=True, order=1, value=45),
    dict(name="RandomPerspective", enabled=True, order=2, value=0.6),
]
spec_B = [
    dict(name="RandomPerspective", enabled=True, order=1, value=0.6),
    dict(name="RandomRotation",    enabled=True, order=2, value=45),
]

def zeige(spec, titel, ax_row, axes):
    torch.manual_seed(123); np.random.seed(123)        # gleicher Seed fuer beide!
    compose = baue_compose(spec)
    for j in range(4):
        arr = augmentiere(kreis, compose)
        axes[ax_row][j].imshow(arr)
        axes[ax_row][j].axis("off")
    axes[ax_row][0].set_ylabel(titel, rotation=0, ha="right", va="center", fontsize=10)

fig, axes = plt.subplots(2, 4, figsize=(9, 4.8))
zeige(spec_A, "A: Rotation\u2192Perspektive", 0, axes)
zeige(spec_B, "B: Perspektive\u2192Rotation", 1, axes)
fig.suptitle("Gleiche Transformationen, gleicher Seed - andere Reihenfolge, anderes Ergebnis", fontsize=11)
fig.tight_layout()
plt.show()

## 7 · 🚀 Brücke zum Transfer-Learning (VGG16 / MobileNet)

Der erzeugte Ordner liegt jetzt im **`ImageFolder`-Format** vor – genau das, was eine Trainings-App
direkt einlesen kann. Die folgende Zelle baut die `DataLoader` und zeigt, wie die **ImageNet-Normalisierung**
*beim Laden* angewandt wird (sie wird **nicht** in die gespeicherten Bilder eingebacken!).

> 🧠 **Merksatz:** Wir speichern rohe `uint8`-RGB-Bilder (0–255). Erst der Loader macht daraus
> `ToTensor()` (→ 0–1) und `Normalize(mean, std)` mit den **ImageNet-Statistiken**, auf die VGG16 und
> MobileNet vortrainiert wurden. Train bekommt die Augmentierungen, Test nur Resize/CenterCrop.

> 📌 Für das eigentliche Transfer-Learning lädst du danach z. B. `torchvision.models.vgg16(weights=...)`
> oder `mobilenet_v2(weights=...)`, frierst die Feature-Schichten ein und ersetzt den Klassifikator-Kopf
> durch einen mit `len(train_ds.classes)` Ausgängen. Das ist die nächste Sitzung – hier steht nur die
> Datenpipeline.

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader

BASIS = "Datasets"
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Train: leichte Live-Augmentierung + Normalisierung (die gespeicherten Bilder sind bereits augmentiert,
#        hier nur noch Standard-Flip/Crop als Beispiel). Test: nur Resize/CenterCrop + Normalisierung.
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(ZIEL_SIZE, scale=(0.8, 1.0), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(ZIEL_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dir = os.path.join(BASIS, "Train")
test_dir  = os.path.join(BASIS, "Test")

if os.path.isdir(train_dir) and any(os.scandir(train_dir)):
    train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
    train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
    print("Klassen (train):", train_ds.classes)
    print("class_to_idx    :", train_ds.class_to_idx)
    print("Train-Bilder    :", len(train_ds))
    xb, yb = next(iter(train_dl))
    print("Batch-Tensor    :", tuple(xb.shape), "| dtype:", xb.dtype, "| Labels:", yb[:8].tolist())

    if os.path.isdir(test_dir) and any(os.scandir(test_dir)):
        test_ds = datasets.ImageFolder(test_dir, transform=test_tf)
        test_dl = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)
        print("Test-Bilder     :", len(test_ds))
    else:
        print("\u26a0\ufe0f Kein Test-Ordner befuellt - bei Einzelbild-Klassen \u22652 Bilder anlegen "
              "oder Demo-Modus nutzen.")
else:
    print("Noch kein Datensatz erzeugt. Oben in der App den Button "
          "'Datensatz erzeugen' druecken, dann diese Zelle erneut ausfuehren.")

### 🧭 Reflexionsfragen für die Studierenden
1. Warum wird **nur** auf den Trainingsdaten augmentiert? Was würde passieren, wenn augmentierte
   Varianten desselben Originalbilds in Train *und* Test landen?
2. VGG16 und MobileNet erwarten **224×224×3**. Welche Rolle spielt `RandomResizedCrop` dabei – und warum
   sorgt das finale `Resize` dafür, dass die Ausgabe *immer* passt?
3. Warum normalisieren wir mit den **ImageNet-Mittelwerten/Standardabweichungen** und nicht mit den
   Statistiken unseres eigenen kleinen Datensatzes?
4. `RandomVerticalFlip` ist bei manchen Klassen label-erhaltend, bei anderen nicht. Nenne je ein Beispiel.
   (Tipp: Ziffern vs. Tierfotos.)
5. Erkläre an deinem eigenen Beispiel, warum `A ≠ B` im Reihenfolge-Experiment gilt.
6. Wir speichern rohe `uint8`-Bilder statt fertig normalisierter Tensoren. Welche **Vorteile** hat das
   für Wiederverwendbarkeit und Speicherplatz?

---
## 📎 Hinweise für die Lehrkraft

**Warum Bildordner statt CSV?** Bei 224×224×3 = 150 528 Werten pro Bild ist eine Pixel-CSV unbrauchbar.
Das `ImageFolder`-Format ist der De-facto-Standard für Transfer-Learning in PyTorch und koppelt sauber
an die nächste Sitzung (Modell laden, Kopf ersetzen, fine-tunen).

**Didaktische Sollbruchstellen / Erweiterungen**
- **Datenleckage** als zentrales Lernziel: Der Quell-basierte Split macht greifbar, *warum* man vor dem
  Augmentieren splittet. Der „Demo-Modus" zeigt bewusst das Gegenteil – ideal zum Diskutieren.
- Eigene Fotos sammeln lassen (≥ 10–20 pro Klasse) → realistisches Mini-Transfer-Learning.
- `ColorJitter` motiviert die Farbdiskussion: Warum hilft das bei Fotos, war aber in der Graustufen-Version
  sinnlos?
- Augmentierungsstärke vs. Genauigkeit: zu starke Augmentierung kann schaden – als Experiment messbar.

**Technik**
- Alle Transformationen laufen auf PIL `'RGB'` → Ausgabe bleibt `uint8`, garantiert 224×224×3.
- Test nutzt das deterministische `Resize(256)`→`CenterCrop(224)`-Rezept (kein Zufall, keine Leckage).
- Die JSON speichert Transformationen als **Daten** (kein pickle): lesbar, sicher, exakt rekonstruierbar.
- `manifest.csv` + `class_index.json` erleichtern Logging, Stratifizierung und Reproduzierbarkeit.
- Leerflächen (durch Rotation/Perspektive) werden mit 0 (schwarz) gefüllt.